In [32]:
import ast
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

In [33]:
ds = load_dataset("TimSchopf/arxiv_categories", "default")

train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()


In [34]:
PHYSICS_LIKE = {
    "gr-qc", "hep-ex", "hep-lat", "hep-ph", "hep-th",
    "nucl-ex", "nucl-th", "quant-ph"
}

In [35]:
train_df.head(2)

,id,title,abstract,categories,creation_date
0,2204.14117,A Comparative Study of Meter Detection Methods...,In order to read meter values from a camera on...,[Computer Science Archive->cs.CV],2022-04-24 13:59:57+00:00
1,2305.19887,The Markov chain embedding problem in a low ju...,We consider the problem of finding the transit...,[Mathematics Archive->math.PR],2023-05-31 14:24:25+00:00


In [36]:
def rewrite_category(categories):
    if len(categories) == 0:
        return None
    return categories[0].split("->")[-1]

def get_category_name(cat):
    if cat is None:
        return None

    if cat.startswith("cs."):
        return "computer_science"
    if cat.startswith("math."):
        return "mathematics"
    if cat.startswith("stat."):
        return "statistics"
    if cat.startswith("q-bio."):
        return "biology"
    if cat.startswith("q-fin."):
        return "finance"
    if cat.startswith("econ."):
        return "economics"
    if cat.startswith("eess."):
        return "electrical_engineering"

    if (
        cat.startswith("physics.")
        or cat.startswith("astro-ph.")
        or cat.startswith("cond-mat.")
        or cat in PHYSICS_LIKE
    ):
        return "physics"

    return None

In [37]:
get_category_name(rewrite_category(train_df["categories"][0]))

'computer_science'

In [38]:
def prepare_df(df):
    df["title"] = df["title"].fillna("").str.strip()
    df["abstract"] = df["abstract"].fillna("").str.strip()

    df["simple_category"] = df["categories"].apply(rewrite_category)
    df["topic"] = df["simple_category"].apply(get_category_name)

    return df

In [39]:
def prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["title"] = df["title"].fillna("").astype(str).str.strip()
    df["abstract"] = df["abstract"].fillna("").astype(str).str.strip()

    df["simple_category"] = df["categories"].apply(rewrite_category)
    df["topic"] = df["simple_category"].apply(get_category_name)

    df = df.dropna(subset=["title", "topic"]).copy()
    df = df[df["title"] != ""].copy()

    if "id" in df.columns:
        df = df.drop_duplicates(subset=["id"], keep="first")

    return df

In [40]:
train_df.head(2)

,id,title,abstract,categories,creation_date
0,2204.14117,A Comparative Study of Meter Detection Methods...,In order to read meter values from a camera on...,[Computer Science Archive->cs.CV],2022-04-24 13:59:57+00:00
1,2305.19887,The Markov chain embedding problem in a low ju...,We consider the problem of finding the transit...,[Mathematics Archive->math.PR],2023-05-31 14:24:25+00:00


In [41]:
prepare_df(train_df).head(2)

,id,title,abstract,categories,creation_date,simple_category,topic
0,2204.14117,A Comparative Study of Meter Detection Methods...,In order to read meter values from a camera on...,[Computer Science Archive->cs.CV],2022-04-24 13:59:57+00:00,cs.CV,computer_science
1,2305.19887,The Markov chain embedding problem in a low ju...,We consider the problem of finding the transit...,[Mathematics Archive->math.PR],2023-05-31 14:24:25+00:00,math.PR,mathematics


In [42]:
train_df = prepare_df(train_df)
val_df = prepare_df(val_df)
test_df = prepare_df(test_df)

In [43]:
label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["topic"])
val_df["label"] = label_encoder.transform(val_df["topic"])
test_df["label"] = label_encoder.transform(test_df["topic"])


id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

In [44]:
train_df["topic"].value_counts()

topic
physics                   81692
mathematics               33816
computer_science          32994
statistics                 1937
electrical_engineering      821
biology                     725
economics                   369
Name: count, dtype: int64

In [45]:
title_only_frac = 0.25

title_only_aug = train_df.sample(frac=title_only_frac, random_state=42).copy()
title_only_aug["abstract"] = ""

train_df_aug = pd.concat([train_df, title_only_aug], ignore_index=True)

print("Original train size:", len(train_df))
print("Augmented train size:", len(train_df_aug))

Original train size: 152354
Augmented train size: 190442


In [46]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df_aug[["title", "abstract", "label"]],
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        val_df[["title", "abstract", "label"]],
        preserve_index=False
    ),
    "test": Dataset.from_pandas(
        test_df[["title", "abstract", "label"]],
        preserve_index=False
    ),
})


In [51]:
model_name = "distilbert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(batch):
    return tokenizer(
        batch["title"],
        batch["abstract"],
        truncation="only_second",
        max_length=256,
    )
    
dataset = dataset.map(tokenize_function, batched=True)
dataset = dataset.remove_columns(["title", "abstract"])
dataset = dataset.rename_column("label", "labels")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_),
    id2label=id2label,
    label2id=label2id,
)

Map:   0%|          | 0/190442 [00:00<?, ? examples/s]

Map:   0%|          | 0/19045 [00:00<?, ? examples/s]

Map:   0%|          | 0/19045 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [52]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [82]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="arxiv_topic_cls",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/home/svyat/Desktop/ШАД/ML2/venv/lib/python3.12/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:310.)
  return F.linear(input, self.weight, self.bias)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.235016,0.183158,0.938462,0.633647
2,0.171453,0.190203,0.941244,0.681421
3,0.118099,0.229502,0.939039,0.694788


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=35709, training_loss=0.1847887158564116, metrics={'train_runtime': 4702.9108, 'train_samples_per_second': 121.483, 'train_steps_per_second': 7.593, 'total_flos': 3.783488262468912e+16, 'train_loss': 0.1847887158564116, 'epoch': 3.0})

In [83]:
save_dir = "Article_classification_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("saved to:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to: Article_classification_model


In [109]:
val_metrics = trainer.evaluate(dataset["validation"])
test_metrics = trainer.evaluate(dataset["test"])

print("Validation metrics:")
print(val_metrics)
print()

print("Test metrics:")
print(test_metrics)

Validation metrics:
{'eval_loss': 0.22950193285942078, 'eval_accuracy': 0.9390391178787083, 'eval_macro_f1': 0.694788382144606, 'eval_runtime': 49.9502, 'eval_samples_per_second': 381.28, 'eval_steps_per_second': 11.932, 'epoch': 3.0}

Test metrics:
{'eval_loss': 0.22953495383262634, 'eval_accuracy': 0.93945917563665, 'eval_macro_f1': 0.7008427731554209, 'eval_runtime': 50.258, 'eval_samples_per_second': 378.944, 'eval_steps_per_second': 11.859, 'epoch': 3.0}


In [62]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_DIR = "Article_classification_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model device: cuda:0


In [10]:
def predict_top(model, tokenizer, device, title: str, abstract: str = "", threshold: float = 0.95,
                max_length: int = 256):
    df = predict_all_probs(model, tokenizer, device, title, abstract, max_length=max_length)

    cumulative = 0.0
    keep_rows = []

    for _, row in df.iterrows():
        keep_rows.append(row)
        cumulative += row["probability"]
        if cumulative >= threshold:
            break

    return pd.DataFrame(keep_rows).reset_index(drop=True)

In [11]:
def predict_all_probs(model, tokenizer, device, title: str, abstract: str = "", max_length: int = 256):
    model.eval()

    inputs = tokenizer(
        title.strip(),
        (abstract or "").strip(),
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=True,
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0]
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

    id2label_local = model.config.id2label

    rows = []
    for idx, p in enumerate(probs):
        rows.append({
            "topic": id2label_local[idx],
            "probability": float(p),
            "probability_percent": round(float(p) * 100, 2),
        })

    return pd.DataFrame(rows).sort_values("probability", ascending=False).reset_index(drop=True)


**Приведем примеры работы нашей модели**

In [17]:
title = "Dual Recurrent Attention Units for Visual Question Answering"

abstract = """
We propose an architecture for VQA which utilizes recurrent layers to generate
visual and textual attention. The model offers a rich joint embedding of visual
and textual features and enables reasoning about relations between several parts
of the image and question.
"""

predict_top(model, tokenizer, device, title, abstract)

,topic,probability,probability_percent
0,computer_science,0.999809,99.98


In [15]:
title = "Observation of Gravitational Waves from a Binary Black Hole Merger"
predict_top(model, tokenizer, device, title)

,topic,probability,probability_percent
0,physics,0.999532,99.95


In [16]:
title = "Phase Transitions in the Two-Dimensional Ising Model"
predict_top(model, tokenizer, device, title)

,topic,probability,probability_percent
0,physics,0.997844,99.78


In [18]:
title = "The Impact of Monetary Policy on Inflation and Unemployment"
abstract = """
We study the effect of monetary policy shocks on inflation and unemployment using
macroeconomic time series data. The analysis focuses on central bank interventions,
interest rates, and labor market responses.
"""

predict_top(model, tokenizer, device, title, abstract)

,topic,probability,probability_percent
0,economics,0.993174,99.32


In [19]:
title = "Market Competition and Firm Productivity in Developing Economies"
predict_top(model, tokenizer, device, title)

,topic,probability,probability_percent
0,economics,0.919075,91.91
1,computer_science,0.065938,6.59


In [20]:
title = "Study of chikens reproduction"
predict_top(model, tokenizer, device, title)

,topic,probability,probability_percent
0,biology,0.760477,76.05
1,physics,0.233220,23.32


In [21]:
title = "Bayesian Inference for Hierarchical Models with Missing Data"

abstract = """
We develop a Bayesian approach for hierarchical statistical models with missing data.
Posterior inference is performed using Markov chain Monte Carlo methods, and uncertainty
quantification is studied through credible intervals and posterior predictive checks.
"""

predict_top(model, tokenizer, device, title, abstract)

,topic,probability,probability_percent
0,statistics,0.99443,99.44


In [22]:
title = "A Statistical Framework for Causal Inference with Observational Data"

abstract = """
We propose a statistical framework for causal inference from observational data.
The methodology combines propensity score adjustment, semiparametric estimation,
and sensitivity analysis for unmeasured confounding.
"""

predict_top(model, tokenizer, device, title, abstract)

,topic,probability,probability_percent
0,statistics,0.98942,98.94


In [24]:
title = "The value of learned societies in the biological sciences: benefits, threats, and futures"
predict_top(model, tokenizer, device, title)

,topic,probability,probability_percent
0,biology,0.98349,98.35


In [25]:
title = "A Scoping Review of Human–Jaguar (Panthera onca) Conflict in Latin America"
predict_top(model, tokenizer, device, title)

,topic,probability,probability_percent
0,biology,0.544161,54.42
1,computer_science,0.348059,34.81
2,economics,0.066328,6.63


Добавим abstract:

In [27]:
title = "A Scoping Review of Human–Jaguar (Panthera onca) Conflict in Latin Americ"

abstract = """
Human–wildlife conflict is a global conservation concern because it threatens human livelihoods and safety and often leads to
the persecution of wildlife. Human–jaguar (Panthera onca) conflict is prevalent throughout the species' range and could
intensify given the projected increases in human population and agricultural production in Latin America. This scoping review
aims to describe current trends, summarize key findings, and identify gaps in the current peer‐reviewed literature, gray literature, and news articles related to human–jaguar conflict (HJC) in Latin America. Four electronic databases (ScienceDirect,
Google Scholar, Web of Science, and the Biological Sciences Database) were used to identify peer‐reviewed articles and gray
literature, and a search engine (Google) was used to identify news articles. Four web‐based searches were performed between
August 2021 and September 2023 in English, Spanish, Portuguese, Dutch, and French to capture results from all countries
within the jaguar's range. Of 422 publications related to HJC, 192 met the inclusion criteria. Selected publications included 89
peer‐reviewed articles, 29 gray literature publications, and 74 news articles published between 1985 and 2023. Most research
was conducted in Brazil and Mexico, used survey or interview methodology, and focused on characterizing conflict and
assessing local communities' perceptions of and attitudes toward jaguars. We identified four major themes and one subtheme:
(1) perceptions, attitudes, and behaviors, (1.1) jaguar killings, (2) attacks on domestic animals, (3) attacks on humans, and
(4) conflict management. While literature related to HJC is extensive, several important gaps remain in (1) management
assessment, (2) policy enforcement, (3) interdisciplinary research, (4) inclusion of a wider range of stakeholders, and
(5) research in countries that harbor a large proportion of the world's jaguars. 
"""

predict_top(model, tokenizer, device, title, abstract)

,topic,probability,probability_percent
0,biology,0.518522,51.85
1,computer_science,0.449615,44.96


Улучшить результат не удалось. Но тем не менее биология все равно является наиболее вероятной темой статьи.

А вот совсем неудачный пример:

In [23]:
title = "Option Pricing Under Stochastic Volatility Models"
abstract = """
We consider option pricing in financial markets under stochastic volatility.
Closed-form approximations and numerical methods are developed for derivative
valuation and risk management.
"""

predict_top(model, tokenizer, device, title, abstract)

,topic,probability,probability_percent
0,mathematics,0.992907,99.29


Мы видим, что наша модель запросто может выдавать 99% уверенность в очень странном ответе. Попробуем откалибровать вероятнсоти.

In [63]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="arxiv_topic_cls",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [64]:
import torch
import torch.nn as nn
import numpy as np

def collect_logits_and_labels(trainer, eval_dataset):
    pred_output = trainer.predict(eval_dataset)
    logits = pred_output.predictions
    labels = pred_output.label_ids

    logits = torch.tensor(logits, dtype=torch.float32)
    labels = torch.tensor(labels, dtype=torch.long)
    return logits, labels


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, logits):
        return logits / self.temperature.clamp(min=1e-6)


def fit_temperature(logits, labels, max_iter=200):
    scaler = TemperatureScaler()
    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.LBFGS(
        scaler.parameters(),
        lr=0.01,
        max_iter=max_iter,
    )

    def closure():
        optimizer.zero_grad()
        loss = criterion(scaler(logits), labels)
        loss.backward()
        return loss

    optimizer.step(closure)

    T = scaler.temperature.item()
    return T

In [65]:
val_logits, val_labels = collect_logits_and_labels(trainer, dataset["validation"])
T = fit_temperature(val_logits, val_labels)

print("Learned temperature:", T)

Learned temperature: 1.556920051574707


Будем использовать эту температуру 

In [66]:
TEMPERATURE = 1.556920051574707

In [67]:
def predict_all_probs(model, tokenizer, device, title: str, abstract: str = "", max_length: int = 256):
    model.eval()

    inputs = tokenizer(
        title.strip(),
        (abstract or "").strip(),
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=True,
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0]
        logits /= TEMPERATURE
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

    id2label_local = model.config.id2label

    rows = []
    for idx, p in enumerate(probs):
        rows.append({
            "topic": id2label_local[idx],
            "probability": float(p),
            "probability_percent": round(float(p) * 100, 2),
        })

    return pd.DataFrame(rows).sort_values("probability", ascending=False).reset_index(drop=True)

In [69]:
title = "Option Pricing Under Stochastic Volatility Models"
abstract = """
We consider option pricing in financial markets under stochastic volatility.
Closed-form approximations and numerical methods are developed for derivative
valuation and risk management.
"""

predict_top(model, tokenizer, device, title, abstract)

,topic,probability,probability_percent
0,mathematics,0.938754,93.88
1,computer_science,0.028206,2.82


стало немного лучше, но не сильно, с 99% вероятность упала до 94%